### Newton's Forward & Backward Interpolation

**Newton's Forward Interpolation** is used to estimate values near the **beginning** of a table of equally spaced data, using forward differences:
$$
P(x) = y_0 + p\,\Delta y_0 + \frac{p(p-1)}{2!}\Delta^2 y_0 + \dots, \qquad p = \frac{x - x_0}{h}
$$

**Newton's Backward Interpolation** is used near the **end** of the table, using backward differences:
$$
P(x) = y_n + q\,\nabla y_n + \frac{q(q+1)}{2!}\nabla^2 y_n + \dots, \qquad q = \frac{x - x_n}{h}
$$

Both methods require **equally spaced** data points (spacing \\(h\\)) and are built from finite-difference tables.


In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

In [2]:
def f(x):
    return np.sin(x)

x_data = np.linspace(0, np.pi, 6)
h = x_data[1] - x_data[0]
y_data = f(x_data)


In [3]:
def difference_table(y_data):
    n = len(y_data)
    table = np.zeros((n, n))
    table[:, 0] = y_data

    print("Forward difference table:")
    header = "y" + "".join([f"   D^{k}y" for k in range(1, n)])
    print(header)

    for j in range(1, n):
        for i in range(n - j):
            table[i, j] = table[i + 1, j - 1] - table[i, j - 1]

    for i in range(n):
        row = " ".join(f"{table[i, j]:9.5f}" for j in range(n - i))
        print(row)

    return table

table = difference_table(y_data)


Forward difference table:
y   D^1y   D^2y   D^3y   D^4y   D^5y
  0.00000   0.58779  -0.22451  -0.13876   0.13876   0.00000
  0.58779   0.36327  -0.36327  -0.00000   0.13876
  0.95106   0.00000  -0.36327   0.13876
  0.95106  -0.36327  -0.22451
  0.58779  -0.58779
  0.00000


In [4]:
def newton_forward(x_data, table, x_query, h):
    n = len(x_data)
    p = (x_query - x_data[0]) / h
    result = table[0, 0]
    p_term = 1.0

    print(f"{'k':>2} {'term':>12} {'cumulative':>14}")
    print("-" * 35)
    print(f"{0:2d} {table[0,0]:12.6f} {result:14.6f}")

    for k in range(1, n):
        p_term *= (p - (k - 1))
        term = p_term * table[0, k] / math.factorial(k)
        result += term
        print(f"{k:2d} {term:12.6f} {result:14.6f}")

    return result

x_query = 0.5
y_interp = newton_forward(x_data, table, x_query, h)
print(f"\nP({x_query}) = {y_interp:.6f}")
print(f"True value f({x_query}) = {f(x_query):.6f}")


 k         term     cumulative
-----------------------------------
 0     0.000000       0.000000
 1     0.467745       0.467745
 2     0.018244       0.485988
 3    -0.004526       0.481462
 4    -0.002494       0.478968
 5     0.000000       0.478968

P(0.5) = 0.478968
True value f(0.5) = 0.479426


In [5]:
plt.figure(figsize=(12, 7), dpi=120)

# Axes lines (Desmos-style)
plt.axhline(0)
plt.axvline(0)

x = np.linspace(0, np.pi, 500)
y_true = f(x)

plt.plot(x, y_true, linewidth=2, label=r"$f(x)=\sin(x)$")
plt.plot(x_data, y_data, 'o', markersize=8, label="Known data points")
plt.scatter([x_query], [y_interp], s=80, zorder=5, color="red", label="Forward interpolated point")

plt.grid(True, alpha=0.4)
plt.xlabel("x", fontsize=12)
plt.ylabel("f(x)", fontsize=12)
plt.title("Newton's Forward Interpolation", fontsize=14)

plt.legend(fontsize=11)
plt.show()
